In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS football.silver

In [0]:
%python
from pyspark.sql import functions as F, types as T

event_schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("match_id", T.LongType()),
    T.StructField("index", T.IntegerType()),
    T.StructField("period", T.IntegerType()),
    T.StructField("minute", T.IntegerType()),
    T.StructField("second", T.IntegerType()),
    T.StructField("type", T.StructType([T.StructField("name", T.StringType())])),
    T.StructField("team", T.StructType([T.StructField("name", T.StringType())])),
    T.StructField("player", T.StructType([T.StructField("name", T.StringType())])),
    T.StructField("location", T.ArrayType(T.DoubleType())),
    T.StructField("shot", T.StructType([
        T.StructField("statsbomb_xg", T.DoubleType()),
        T.StructField("outcome", T.StructType([T.StructField("name", T.StringType())])),
    ])),
])

def run_silver():
    return (spark.readStream.table("football.bronze.events_raw")
        .select(F.from_json("payload", event_schema).alias("e"), "bronze_ts")
        .select(
            F.col("e.id").alias("id"),
            F.col("e.match_id").alias("match_id"),
            F.col("e.index").alias("event_index"),
            F.col("e.period").alias("period"),
            F.col("e.minute").alias("minute"),
            F.col("e.second").alias("second"),
            F.col("e.type.name").alias("event_type"),
            F.col("e.team.name").alias("team"),
            F.col("e.player.name").alias("player"),
            F.col("e.location")[0].alias("x"),
            F.col("e.location")[1].alias("y"),
            F.col("e.shot.statsbomb_xg").alias("xg"),
            F.col("e.shot.outcome.name").alias("shot_outcome"),
            F.col("bronze_ts"))
        .filter("id IS NOT NULL AND match_id IS NOT NULL")
        .dropDuplicates(["id"])
        .writeStream
        .option("checkpointLocation", "/Volumes/football/bronze/chk/silver_events")
        .trigger(availableNow=True)
        .toTable("football.silver.events"))

In [0]:
%python
q = run_silver()
q.awaitTermination()
print(q.lastProgress)

In [0]:
%sql
SELECT event_type, COUNT(*) AS n
FROM football.silver.events
GROUP BY event_type
ORDER BY n DESC

In [0]:
%sql
SELECT team, minute, player, ROUND(xg, 2) AS xg, shot_outcome
FROM football.silver.events
WHERE event_type = 'Shot'
ORDER BY minute
LIMIT 10